# 12 - Fine-tuning IndoBERTweet

Fine-tune IndoBERTweet (`indolem/indobertweet-base-uncased`) buat 2 skema (**binary** & **3class**),
pakai data hasil tahap 11 (`output/11_finalisasi_data_training/<skema>/`).

**Logic split validasi internal pakai versi yang sudah diperbaiki** (anti-leakage): validasi internal
(early-stopping) SELALU diambil cuma dari baris `asli`, baris sintesis/dummy selalu masuk training.
Kalau validasi ikut kecampur sintesis, itu bisa "bocor" -- sintesis adalah beberapa variasi paraphrase
dari komentar yang SAMA, jadi kalau 1 variasi masuk training dan variasi lain masuk validasi, model
keliatan jago padahal cuma hafalan pola kalimat.

**Aturan lain:**
- `test.csv` adalah HOLD-OUT SUNGGUHAN (dipisah dari awal tahap 10), tidak pernah disentuh augmentasi.
- CV (5-fold, `RUN_CV`) cuma buat development/tuning; hasil akhir yang dilaporkan adalah SATU model
  final (dilatih pakai SELURUH train.csv) dievaluasi SEKALI ke test.csv.
- `class_weights` dihitung dari distribusi train.csv yang dipakai (formula 'balanced').

Model: `indolem/indobertweet-base-uncased`. `max_len=56` (comparable sama baseline S0/S1 di skripsi).

**WAJIB dijalankan di mesin dengan GPU + akses internet (HuggingFace hub)** -- belum pernah berhasil
dijalankan penuh di environment pengembangan (sandbox tanpa GPU/akses hub).

In [ ]:
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

# ============================ KONFIGURASI ============================
FINAL_OUTPUT_ROOT = Path.cwd() / "output"
DATA_DIR = FINAL_OUTPUT_ROOT / "11_finalisasi_data_training"
OUTPUT_DIR = FINAL_OUTPUT_ROOT / "12_finetune_indobert"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS_DIR = OUTPUT_DIR / "models"
RESULTS_DIR = OUTPUT_DIR / "hasil_cv"

SCHEMES = ["binary", "3class"]
EPOCHS = 4
LR = 2e-5
BATCH_TRAIN = 16       # turunkan ke 8 kalau CUDA out of memory
BATCH_EVAL = 32
MAX_LEN = 56
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOP_PATIENCE = 2
VAL_SIZE_INTERNAL = 0.1   # dipecah dari baris asli, buat early stopping
RUN_CV = False             # False -> skip 5-fold CV, langsung ke model final (lebih cepet).
SAVE_MODELS = False        # True -> simpan model fold terbaik + model final ke models/<skema>/
SEED = 42
MODEL_NAME = "indolem/indobertweet-base-uncased"

LABEL2ID = {
    "binary": {"non-negatif": 0, "negatif": 1},
    "3class": {"negatif": 0, "netral": 1, "positif": 2},
}
LABEL_COL = {"binary": "label_name", "3class": "label_final"}
print("Config siap.")

## Fungsi: load data & logic split (anti-leakage, tanpa torch)

In [ ]:
def load_train_test(scheme):
    """Load train.csv & test.csv buat 1 skema, mapping label ke label_id (int), dan pastikan
    kolom 'fold' ada (string, blank buat baris non-asli)."""
    data_dir = DATA_DIR / scheme
    train_path = data_dir / "train.csv"
    test_path = data_dir / "test.csv"
    for p in (train_path, test_path):
        if not p.exists():
            raise FileNotFoundError(f"{p} tidak ditemukan. Jalankan tahap 11 dulu.")

    train_df = pd.read_csv(train_path, dtype=str, keep_default_na=False, encoding="utf-8")
    test_df = pd.read_csv(test_path, dtype=str, keep_default_na=False, encoding="utf-8")

    label_col = LABEL_COL[scheme]
    label2id = LABEL2ID[scheme]

    for nama, df in [("train", train_df), ("test", test_df)]:
        required = {"text", "sumber", label_col}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"{scheme}/{nama}.csv kurang kolom: {missing}")
        tidak_dikenal = set(df[label_col].unique()) - set(label2id.keys())
        if tidak_dikenal:
            raise ValueError(f"{scheme}/{nama}.csv punya label tak dikenal: {tidak_dikenal}")
        df["label_id"] = df[label_col].map(label2id)

    if "fold" not in train_df.columns:
        train_df["fold"] = ""

    return train_df, test_df


def compute_class_weights(label_ids, n_classes):
    """Formula 'balanced' standar: weight_c = n_samples / (n_classes * count_c).
    Kelas yang jarang (misal negatif) dapet bobot loss lebih besar."""
    label_ids = np.asarray(label_ids)
    counts = np.array([(label_ids == c).sum() for c in range(n_classes)])
    if (counts == 0).any():
        raise ValueError(f"Ada kelas dengan 0 anggota: counts={counts.tolist()}")
    n_samples = len(label_ids)
    return n_samples / (n_classes * counts)

In [ ]:
def split_asli_only_validation(df, label_col="label_id", val_size=VAL_SIZE_INTERNAL, seed=42):
    """Pisahin validasi internal (buat early-stopping) HANYA dari baris asli; sisanya (asli
    yang ga kepilih + SEMUA sintesis/dummy) jadi training.

    Ini penting: kalau validasi ikut kecampur sintesis, itu bisa "bocor" -- sintesis adalah
    beberapa variasi paraphrase dari komentar YANG SAMA, jadi kalau 1 variasi masuk training
    dan variasi lain (dari komentar asal yang sama) masuk validasi, model keliatan jago pas
    divalidasi padahal cuma hafalan pola kalimat, bukan belajar sentimen beneran. Ini bikin
    early-stopping salah berhenti (telat), model jadi overfit ke training set (termasuk ke
    sintesis-nya sendiri) dan jeblok pas ketemu data asli yang beneran baru (test.csv).

    Fallback ke stratified split biasa kalau df ga punya baris asli sama sekali."""
    asli_mask = df["sumber"] == "asli"
    asli_df = df[asli_mask]
    non_asli_df = df[~asli_mask]

    if len(asli_df) == 0:
        train_final_df, val_final_df = train_test_split(
            df, test_size=val_size, random_state=seed, stratify=df[label_col],
        )
        return train_final_df.reset_index(drop=True), val_final_df.reset_index(drop=True)

    try:
        asli_train_df, asli_val_df = train_test_split(
            asli_df, test_size=val_size, random_state=seed, stratify=asli_df[label_col],
        )
    except ValueError:
        asli_train_df, asli_val_df = train_test_split(asli_df, test_size=val_size, random_state=seed)
    train_final_df = pd.concat([asli_train_df, non_asli_df], ignore_index=True)
    val_final_df = asli_val_df.reset_index(drop=True)
    return train_final_df, val_final_df


def split_fold_for_cv(df, fold_value, label_col="label_id", seed=42, val_size=VAL_SIZE_INTERNAL):
    """1 iterasi CV: pisahin held-out validation (cuma baris asli dgn fold == fold_value) dari
    sisanya (baris asli fold lain + SEMUA sintesis/dummy), lalu dipecah lagi jadi
    train_sub_df/val_internal_df buat early-stopping (val_internal_df-nya JUGA cuma dari baris
    asli, lihat split_asli_only_validation).

    Return: (train_sub_df, val_internal_df, held_out_df)"""
    held_out_mask = (df["sumber"] == "asli") & (df["fold"] == str(fold_value))
    held_out_df = df[held_out_mask].reset_index(drop=True)
    remain_df = df[~held_out_mask].reset_index(drop=True)

    if len(held_out_df) == 0:
        raise ValueError(f"Ga ada baris asli dengan fold == {fold_value}. Cek kolom fold di train.csv.")

    train_sub_df, val_internal_df = split_asli_only_validation(
        remain_df, label_col=label_col, val_size=val_size, seed=seed,
    )
    return train_sub_df, val_internal_df, held_out_df


def available_folds(train_df):
    """Daftar nilai fold yang valid (cuma dari baris asli, exclude blank)."""
    folds = train_df.loc[train_df["sumber"] == "asli", "fold"]
    return sorted({f for f in folds if f.strip() != ""}, key=int)


print("Fungsi logic (anti-leakage split, class weight) siap -- bisa dicek tanpa torch/GPU.")

## Setup ML (torch/transformers) -- WAJIB GPU + akses HuggingFace hub

In [ ]:
def _lazy_import_ml():
    import torch
    from torch.utils.data import Dataset
    from transformers import (
        AutoModelForSequenceClassification, AutoTokenizer, EarlyStoppingCallback,
        Trainer, TrainingArguments,
    )
    return torch, Dataset, AutoModelForSequenceClassification, AutoTokenizer, EarlyStoppingCallback, Trainer, TrainingArguments


def build_dataset_class():
    torch, Dataset, *_ = _lazy_import_ml()

    class CommentDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_len):
            enc = tokenizer(list(texts), truncation=True, padding="max_length",
                             max_length=max_len, return_tensors="pt")
            self.input_ids = enc["input_ids"]
            self.attention_mask = enc["attention_mask"]
            self.labels = torch.tensor(np.asarray(labels), dtype=torch.long)

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            return {"input_ids": self.input_ids[idx], "attention_mask": self.attention_mask[idx],
                     "labels": self.labels[idx]}

    return CommentDataset


def build_weighted_trainer_class():
    torch, _, _, _, _, Trainer, _ = _lazy_import_ml()

    class WeightedTrainer(Trainer):
        def __init__(self, *args, class_weights=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.class_weights = class_weights

        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            logits = outputs.logits
            weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
            loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    return WeightedTrainer


def make_compute_metrics(label_names):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        out = {
            "accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        }
        per_class = f1_score(labels, preds, average=None, zero_division=0,
                              labels=list(range(len(label_names))))
        for name, f1_val in zip(label_names, per_class):
            out[f"f1_{name}"] = f1_val
        return out
    return compute_metrics


print("Fungsi dataset/trainer siap.")

In [ ]:
def train_one_model(train_sub_df, val_df, label_names, class_weight_vec, tokenizer,
                     model_name, max_len, out_dir, seed_offset=0):
    """Latih 1 model IndoBERTweet. Return trainer (bisa dipakai predict lagi)."""
    torch, _, AutoModelForSequenceClassification, _, EarlyStoppingCallback, _, TrainingArguments = _lazy_import_ml()
    CommentDataset = build_dataset_class()
    WeightedTrainer = build_weighted_trainer_class()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    n_labels = len(label_names)

    train_ds = CommentDataset(train_sub_df["text"], train_sub_df["label_id"], tokenizer, max_len)
    val_ds = CommentDataset(val_df["text"], val_df["label_id"], tokenizer, max_len)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=n_labels)

    total_steps = (len(train_sub_df) // BATCH_TRAIN + 1) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    args = TrainingArguments(
        output_dir=str(out_dir / "checkpoints"),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup_steps,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        fp16=(device == "cuda"),
        logging_steps=50,
        seed=SEED + seed_offset,
        report_to="none",
        disable_tqdm=False,
    )

    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=make_compute_metrics(label_names), class_weights=class_weight_vec,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
    )
    trainer.train()
    return trainer


def evaluate_on(trainer, eval_df, label_names, tokenizer, max_len):
    CommentDataset = build_dataset_class()
    ds = CommentDataset(eval_df["text"], eval_df["label_id"], tokenizer, max_len)
    pred_out = trainer.predict(ds)
    preds = np.argmax(pred_out.predictions, axis=-1)
    labels_true = eval_df["label_id"].astype(int).to_numpy()
    report = classification_report(labels_true, preds, target_names=label_names,
                                    output_dict=True, zero_division=0)
    cm = confusion_matrix(labels_true, preds, labels=list(range(len(label_names))))
    return report, cm.tolist()


print("Fungsi train_one_model() & evaluate_on() siap.")

## Fungsi per-tahap: setup run, latih 1 skema, simpan hasil

In [ ]:
def setup_run():
    torch, *_ = _lazy_import_ml()
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)
    if device == "cuda":
        print("GPU   :", torch.cuda.get_device_name(0))
    else:
        print("PERHATIAN: tidak terdeteksi GPU. Training akan sangat lambat di CPU.")
    return device


def run_scheme(scheme, device, all_fold_results, all_fold_reports, final_results):
    """Latih (opsional CV + model final) & evaluasi 1 skema ('binary' atau '3class'),
    nambahin hasilnya ke all_fold_results/all_fold_reports/final_results (mutasi in-place)."""
    torch, _, AutoModelForSequenceClassification, AutoTokenizer, _, _, _ = _lazy_import_ml()

    label2id = LABEL2ID[scheme]
    label_names = [name for name, _ in sorted(label2id.items(), key=lambda kv: kv[1])]
    n_labels = len(label_names)

    train_df, test_df = load_train_test(scheme)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    print(f"\n{'=' * 70}\n  SKEMA: {scheme}  ({n_labels} kelas)\n{'=' * 70}")
    print(f"  train total: {len(train_df)}  (asli={len(train_df[train_df.sumber == 'asli'])}, "
          f"sintesis/dummy={len(train_df[train_df.sumber != 'asli'])})")
    print(f"  test total : {len(test_df)}  (semua asli, hold-out sungguhan)")

    folds = available_folds(train_df)
    print(f"  fold tersedia (dari baris asli): {folds}")

    # ------------------------- CV (buat development/tuning) -------------------------
    if not RUN_CV:
        print("  RUN_CV=False -> skip 5-fold CV, langsung ke model final.")
    for fold in (folds if RUN_CV else []):
        t0 = time.time()
        train_sub_df, val_internal_df, held_out_df = split_fold_for_cv(train_df, fold, seed=SEED + int(fold))
        class_weight_vec_np = compute_class_weights(train_sub_df["label_id"], n_labels)
        class_weight_vec = torch.tensor(class_weight_vec_np, dtype=torch.float)

        out_dir = MODELS_DIR / scheme / f"cv_fold_{fold}"
        trainer = train_one_model(train_sub_df, val_internal_df, label_names, class_weight_vec,
                                   tokenizer, MODEL_NAME, MAX_LEN, out_dir, seed_offset=int(fold))

        report, cm = evaluate_on(trainer, held_out_df, label_names, tokenizer, MAX_LEN)
        row = {
            "scheme": scheme, "fold": fold,
            "n_train": len(train_sub_df), "n_val_internal": len(val_internal_df),
            "n_held_out_asli": len(held_out_df),
            "accuracy": report["accuracy"], "f1_macro": report["macro avg"]["f1-score"],
        }
        for name in label_names:
            row[f"f1_{name}"] = report[name]["f1-score"]
            row[f"precision_{name}"] = report[name]["precision"]
            row[f"recall_{name}"] = report[name]["recall"]
        all_fold_results.append(row)
        all_fold_reports[(scheme, fold)] = {"report": report, "confusion_matrix": cm, "label_names": label_names}
        print(f"  [fold {fold}] f1_macro={row['f1_macro']:.3f} | "
              + " | ".join(f"f1_{n}={row[f'f1_{n}']:.3f}" for n in label_names)
              + f" | waktu={(time.time() - t0) / 60:.1f} menit")

        if SAVE_MODELS:
            out_dir.mkdir(parents=True, exist_ok=True)
            trainer.save_model(str(out_dir))
            tokenizer.save_pretrained(str(out_dir))

        del trainer
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    # ------------------------- Model FINAL (evaluasi sekali ke test.csv) -------------------------
    print(f"\n  --- Melatih model FINAL {scheme} (seluruh train.csv) ---")
    class_weight_vec_np = compute_class_weights(train_df["label_id"], n_labels)
    class_weight_vec = torch.tensor(class_weight_vec_np, dtype=torch.float)

    train_sub_df, val_internal_df = split_asli_only_validation(
        train_df, label_col="label_id", val_size=VAL_SIZE_INTERNAL, seed=SEED,
    )
    out_dir_final = MODELS_DIR / scheme / "final"
    trainer_final = train_one_model(train_sub_df, val_internal_df, label_names, class_weight_vec,
                                     tokenizer, MODEL_NAME, MAX_LEN, out_dir_final)
    report_test, cm_test = evaluate_on(trainer_final, test_df, label_names, tokenizer, MAX_LEN)
    print(f"  [FINAL -> test.csv] accuracy={report_test['accuracy']:.3f} "
          f"f1_macro={report_test['macro avg']['f1-score']:.3f}")
    for name in label_names:
        print(f"    f1_{name} = {report_test[name]['f1-score']:.3f}")

    print("  --- Evaluasi model final ke TRAIN penuh (cek overfitting) ---")
    report_train, cm_train = evaluate_on(trainer_final, train_df, label_names, tokenizer, MAX_LEN)
    print(f"  [FINAL -> train.csv] accuracy={report_train['accuracy']:.3f} "
          f"f1_macro={report_train['macro avg']['f1-score']:.3f}")

    final_results[scheme] = {
        "label_names": label_names,
        "train": {"report": report_train, "confusion_matrix": cm_train},
        "test": {"report": report_test, "confusion_matrix": cm_test},
    }

    if SAVE_MODELS:
        out_dir_final.mkdir(parents=True, exist_ok=True)
        trainer_final.save_model(str(out_dir_final))
        tokenizer.save_pretrained(str(out_dir_final))

    del trainer_final, tokenizer
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()


def save_results(all_fold_results, all_fold_reports, final_results):
    if all_fold_results:
        results_df = pd.DataFrame(all_fold_results)
        results_df.to_csv(RESULTS_DIR / "cv_results_per_fold.csv", index=False)
        metric_cols = [c for c in results_df.columns if c.startswith(("accuracy", "f1_", "precision_", "recall_"))]
        results_df.groupby("scheme")[metric_cols].agg(["mean", "std"]).to_csv(RESULTS_DIR / "cv_results_summary.csv")
    else:
        print("RUN_CV=False -> ga ada hasil CV yang disimpan (cv_results_per_fold.csv/summary.csv dilewat).")

    with open(RESULTS_DIR / "cv_reports_detail.json", "w", encoding="utf-8") as f:
        json.dump({f"{s}_fold{i}": v for (s, i), v in all_fold_reports.items()}, f, indent=2, ensure_ascii=False)

    out_final_train_test = RESULTS_DIR / "final_train_vs_test.json"
    with open(out_final_train_test, "w", encoding="utf-8") as f:
        json.dump(final_results, f, indent=2, ensure_ascii=False)

    print(f"\nTersimpan di {RESULTS_DIR}:")
    print(" - cv_results_per_fold.csv     (metrik CV per fold, held-out cuma dari baris asli)")
    print(" - cv_results_summary.csv      (mean +/- std per skema, dari CV)")
    print(" - cv_reports_detail.json      (classification_report + confusion matrix lengkap per fold)")
    print(" - final_train_vs_test.json    (HASIL AKHIR: model full-train dievaluasi ke train.csv DAN test.csv)")


print("Fungsi setup_run() / run_scheme() / save_results() siap.")

## Jalankan: setup (device, seed) -- skip otomatis kalau hasil akhir sudah ada

In [ ]:
_out_final_train_test = RESULTS_DIR / "final_train_vs_test.json"
if _out_final_train_test.exists():
    print(f"[Finetune] {_out_final_train_test} sudah ada -> training di-skip (hapus file ini kalau mau re-train).")
    device = None
else:
    device = setup_run()

all_fold_results, all_fold_reports, final_results = [], {}, {}

## Latih skema BINARY (non-negatif vs negatif)

In [ ]:
if device is not None:
    run_scheme("binary", device, all_fold_results, all_fold_reports, final_results)

## Latih skema 3CLASS (negatif / netral / positif)

In [ ]:
if device is not None:
    run_scheme("3class", device, all_fold_results, all_fold_reports, final_results)

## Simpan hasil akhir (CV summary + classification report + confusion matrix)

In [ ]:
if device is not None:
    save_results(all_fold_results, all_fold_reports, final_results)

## Lihat hasil -- tabel classification report, confusion matrix, gap train vs test

Baca `hasil_cv/final_train_vs_test.json` (hasil cell di atas, atau dari run sebelumnya) dan tampilin
tabelnya rapi. Bisa dijalankan terpisah tanpa harus re-run training di atas.

In [ ]:
def _tabel_classification_report(report, label_names):
    baris = []
    for name in label_names:
        r = report[name]
        baris.append({"kelas": name, "precision": r["precision"], "recall": r["recall"],
                      "f1-score": r["f1-score"], "support": int(r["support"])})
    for key in ("macro avg", "weighted avg"):
        r = report[key]
        baris.append({"kelas": key, "precision": r["precision"], "recall": r["recall"],
                      "f1-score": r["f1-score"], "support": int(r["support"])})
    baris.append({"kelas": "accuracy", "precision": "", "recall": "", "f1-score": report["accuracy"],
                  "support": int(report["macro avg"]["support"])})
    return pd.DataFrame(baris).set_index("kelas")


def _tabel_perbandingan_train_test(report_train, report_test, label_names):
    baris = []
    for name in label_names + ["macro avg", "weighted avg"]:
        rt, rv = report_train[name], report_test[name]
        baris.append({
            "kelas": name,
            "precision_train": rt["precision"], "precision_test": rv["precision"],
            "recall_train": rt["recall"], "recall_test": rv["recall"],
            "f1_train": rt["f1-score"], "f1_test": rv["f1-score"],
            "gap_f1 (test - train)": rv["f1-score"] - rt["f1-score"],
        })
    baris.append({
        "kelas": "accuracy", "precision_train": "", "precision_test": "", "recall_train": "", "recall_test": "",
        "f1_train": report_train["accuracy"], "f1_test": report_test["accuracy"],
        "gap_f1 (test - train)": report_test["accuracy"] - report_train["accuracy"],
    })
    return pd.DataFrame(baris).set_index("kelas")


def _tabel_confusion_matrix(cm, label_names):
    idx = [f"asli: {n}" for n in label_names]
    cols = [f"prediksi: {n}" for n in label_names]
    return pd.DataFrame(cm, index=idx, columns=cols)


def cetak_laporan_skema(scheme, data, float_fmt="{:.3f}"):
    label_names = data["label_names"]
    report_train, report_test = data["train"]["report"], data["test"]["report"]
    cm_train, cm_test = data["train"]["confusion_matrix"], data["test"]["confusion_matrix"]

    pd.set_option("display.float_format", float_fmt.format)

    print(f"\n{'=' * 70}\nSKEMA: {scheme.upper()}\n{'=' * 70}")

    print("\n-- Classification report TRAIN --")
    print(_tabel_classification_report(report_train, label_names).to_string())

    print("\n-- Classification report TEST --")
    print(_tabel_classification_report(report_test, label_names).to_string())

    print("\n-- Perbandingan TRAIN vs TEST (gap F1 negatif besar = overfitting) --")
    print(_tabel_perbandingan_train_test(report_train, report_test, label_names).to_string())

    print("\n-- Confusion matrix TRAIN --")
    print(_tabel_confusion_matrix(cm_train, label_names).to_string())

    print("\n-- Confusion matrix TEST --")
    print(_tabel_confusion_matrix(cm_test, label_names).to_string())

    gap_acc = report_test["accuracy"] - report_train["accuracy"]
    gap_f1 = report_test["macro avg"]["f1-score"] - report_train["macro avg"]["f1-score"]
    print(f"\nGap accuracy (test - train): {gap_acc:+.3f}")
    print(f"Gap F1-macro (test - train): {gap_f1:+.3f}")
    if gap_f1 < -0.1:
        print("  -> PERHATIAN: gap > 0.1, kemungkinan model overfit ke training set.")
    else:
        print("  -> gap wajar, model keliatan generalize dengan oke.")


_path_hasil = RESULTS_DIR / "final_train_vs_test.json"
if not _path_hasil.exists():
    print(f"[Stats] {_path_hasil} belum ada. Jalankan cell training di atas dulu sampai selesai.")
else:
    with open(_path_hasil, encoding="utf-8") as f:
        hasil = json.load(f)

### Skema BINARY

In [ ]:
if _path_hasil.exists():
    cetak_laporan_skema("binary", hasil["binary"])

### Skema 3CLASS

In [ ]:
if _path_hasil.exists():
    cetak_laporan_skema("3class", hasil["3class"])